# Assignment 2: NLP
**PDF Question Answering Tool - ( Tarek Alam Bhuiyan - 1000129291 )**
#---------------------------------------------

This notebook builds a question answering tool that answers questions from PDF documents.

**Pipeline**
1. Extract text from PDFs (`pypdf`)
2. Clean noisy text (the "Hard" PDF has junk symbol blocks)
3. Split text into overlapping passages
4. Retrieve the most relevant passages for a question (TF-IDF + cosine similarity)
5. Extract the answer span with a pretrained extractive QA model (`deepset/roberta-base-squad2`)
6. Read questions from `Questions.pdf` and answer them for both documents

Kept the three PDF paths in the config cell, then ran all cells.

## 1. Install and import

In [37]:
!pip install -q pypdf transformers torch scikit-learn pandas

In [52]:
import torch
# Force PyTorch backend detection in transformers pipeline registry
if torch.cuda.is_available():
    _ = torch.tensor([1.0]).cuda()

import re
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

## 2. Config


In [49]:
EASY_PDF = "/content/Test_PDF_Easy.pdf"
HARD_PDF = "/content/Test_PDF_Hard.pdf"
QUESTIONS_PDF = "/content/Questions.pdf"

MODEL_NAME = "deepset/roberta-base-squad2"
TOP_K = 3            # passages passed to the QA model
WINDOW = 3           # sentences per passage
STRIDE = 1           # sentence step between passages

## 3. PDF text extraction and cleaning

The Hard PDF has long blocks of junk symbols (`$`, `%`, `&`, `(`, `)`, `_`) inserted in the middle of sentences, and also line breaks inside sentences. The cleaner removes those blocks and normalises whitespace.

In [41]:
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    pages = [(page.extract_text() or "") for page in reader.pages]
    return "\n".join(pages)

# A junk block = a long run of noise symbols, possibly broken by line breaks
NOISE_RE = re.compile(r"(?:[_$%&()]\s*){5,}\.?")

def clean_text(text):
    text = NOISE_RE.sub(" ", text)          # remove symbol blocks
    text = re.sub(r"\s+", " ", text)        # join broken lines, collapse spaces
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)  # no space before punctuation
    return text.strip()

raw_easy = extract_text(EASY_PDF)
raw_hard = extract_text(HARD_PDF)

easy_text = clean_text(raw_easy)
hard_text = clean_text(raw_hard)

print("EASY PDF (cleaned):\n", easy_text[:600], "...\n")
print("HARD PDF (cleaned):\n", hard_text[:600], "...")

EASY PDF (cleaned):
 Questions: 1. What is the capital city of Bangladesh? 2. Which river is considered the lifeline of Bangladesh? 3. In which year did Bangladesh gain independence? 4. What is the official language of Bangladesh? 5. What is the name of the world’s largest mangrove forest located in Bangladesh? 6. Which currency is used in Bangladesh? What is the national animal of Bangladesh? ...

HARD PDF (cleaned):
 Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and

## 4. Build the QA tool

- `split_sentences`: simple regex sentence splitter
- `make_passages`: sliding window of sentences, so answers that span a sentence boundary are still found
- `PDFQuestionAnswerer`: retrieves top passages with TF-IDF, runs the extractive QA model on each, and returns the highest scoring answer

In [44]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    return [p.strip() for p in parts if p.strip()]

def make_passages(text, window=WINDOW, stride=STRIDE):
    sents = split_sentences(text)
    if len(sents) <= window:
        return [" ".join(sents)]
    passages = []
    for i in range(0, len(sents) - window + 1, stride):
        passages.append(" ".join(sents[i:i + window]))
    return passages


class PDFQuestionAnswerer:
    def __init__(self, pdf_paths, model_name=MODEL_NAME, top_k=TOP_K):
        self.top_k = top_k
        # Load the model and tokenizer directly
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.passages = []
        for path in pdf_paths:
            self.passages += make_passages(clean_text(extract_text(path)))
        # drop duplicate passages (the same info may exist in several PDFs)
        self.passages = list(dict.fromkeys(self.passages))
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform(self.passages)

    def retrieve(self, question):
        q_vec = self.vectorizer.transform([question])
        scores = cosine_similarity(q_vec, self.matrix).ravel()
        top_idx = scores.argsort()[::-1][: self.top_k]
        return [self.passages[i] for i in top_idx]

    def answer(self, question):
        best = {"answer": "No answer found", "score": 0.0, "context": ""}
        for ctx in self.retrieve(question):
            # Manually run the QA model directly bypassing pipeline()
            inputs = self.tokenizer(question, ctx, return_tensors="pt", truncation=True, max_length=512)
            with torch.no_grad():
                outputs = self.model(**inputs)

            start_logits = outputs.start_logits
            end_logits = outputs.end_logits

            start_idx = torch.argmax(start_logits)
            end_idx = torch.argmax(end_logits) + 1

            # Convert token indices back to string answer
            answer_tokens = inputs.input_ids[0][start_idx:end_idx]
            ans = self.tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

            # Convert logits to a pseudo-confidence score
            score = float(torch.softmax(start_logits, dim=-1)[0][start_idx] * torch.softmax(end_logits, dim=-1)[0][end_idx - 1])

            if ans and score > best["score"]:
                best = {"answer": ans.strip(" ."), "score": score, "context": ctx}
        return best

## 5. Read the questions from `Questions.pdf`

A numbered line can contain more than one question (e.g. question 6), so each `?` ends a separate question.

In [55]:
def load_questions(pdf_path):
    raw_text = extract_text(pdf_path)
    # Normalize whitespace first: replace all newlines/tabs with spaces so questions are unified
    cleaned_text = re.sub(r"\s+", " ", raw_text).strip()

    # Split by common question patterns or numbering to handle multi-question lines
    # e.g., '1. What is...? 2. Which is...?'
    raw_questions = re.findall(r"[^?]+\?", cleaned_text)

    questions = []
    for q in raw_questions:
        # Clean numbering off the start (e.g., 'Questions: 1.', '1. ', '2) ')
        q_clean = re.sub(r"^.*?\d+[.)]\s*", "", q.strip())
        # Clean any remaining 'Questions:' headers or random noise
        q_clean = re.sub(r"(?i)^\s*questions:\s*", "", q_clean)
        q_clean = q_clean.strip()
        if len(q_clean) > 5:
            questions.append(q_clean)

    return questions

questions = load_questions(QUESTIONS_PDF)
for i, q in enumerate(questions, 1):
    print(f"{i}. {q}")

1. What is the capital city of Bangladesh?
2. Which river is considered the lifeline of Bangladesh?
3. In which year did Bangladesh gain independence?
4. What is the official language of Bangladesh?
5. What is the name of the world’s largest mangrove forest located in Bangladesh?
6. Which currency is used in Bangladesh?
7. What is the national animal of Bangladesh?


## 6. Answer the questions
One tool is built per PDF so each document is evaluated on its own, and a third tool uses both PDFs together.

In [46]:
def run_qa(pdf_paths, questions, label):
    tool = PDFQuestionAnswerer(pdf_paths)
    rows = []
    for i, q in enumerate(questions, 1):
        res = tool.answer(q)
        rows.append({"No": i, "Question": q, "Answer": res["answer"], "Confidence": round(res["score"], 3)})
    df = pd.DataFrame(rows)
    print(f"\n===== {label} =====")
    display(df)
    return df

df_easy = run_qa([EASY_PDF], questions, "Easy PDF")
df_hard = run_qa([HARD_PDF], questions, "Hard PDF")
df_both = run_qa([EASY_PDF, HARD_PDF], questions, "Both PDFs combined")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

...


===== Hard PDF =====


""


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Both PDFs combined =====


""


In [47]:
def run_qa(pdf_paths, questions, label):
    tool = PDFQuestionAnswerer(pdf_paths)
    rows = []
    for i, q in enumerate(questions, 1):
        res = tool.answer(q)
        rows.append({"No": i, "Question": q, "Answer": res["answer"], "Confidence": round(res["score"], 3)})
    df = pd.DataFrame(rows)
    print(f"\n===== {label} =====")
    display(df)
    return df

df_easy = run_qa([EASY_PDF], questions, "Easy PDF")
df_hard = run_qa([HARD_PDF], questions, "Hard PDF")
df_both = run_qa([EASY_PDF, HARD_PDF], questions, "Both PDFs combined")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Easy PDF =====


""


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Hard PDF =====


""


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Both PDFs combined =====


""


## 7. Compare Easy vs Hard PDF results

In [56]:
import pandas as pd

# Run the updated pipeline using the correctly parsed full questions
try:
    # 1. Ensure the correct questions are loaded
    questions = load_questions(QUESTIONS_PDF)
    print(f"Loaded {len(questions)} clean questions. Running QA predictions...")

    # 2. Re-run QA models
    df_easy = run_qa([EASY_PDF], questions, "Easy PDF")
    df_hard = run_qa([HARD_PDF], questions, "Hard PDF")
    df_both = run_qa([EASY_PDF, HARD_PDF], questions, "Both PDFs combined")

    # 3. Create and display clean comparison table
    comparison = pd.DataFrame({
        "Question": df_easy["Question"],
        "Easy PDF": df_easy["Answer"],
        "Hard PDF": df_hard["Answer"],
        "Combined": df_both["Answer"],
    })
    pd.set_option("display.max_colwidth", 80)
    display(comparison)
except Exception as e:
    print(f"An error occurred during execution: {e}")

Loaded 7 clean questions. Running QA predictions...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Easy PDF =====


,No,Question,Answer,Confidence
0,1,What is the capital city of Bangladesh?,Dhaka,0.957
1,2,Which river is considered the lifeline of Bangladesh?,Padma River,0.828
2,3,In which year did Bangladesh gain independence?,1971,0.993
3,4,What is the official language of Bangladesh?,Bengali,0.977
4,5,What is the name of the world’s largest mangrove forest located in Bangladesh?,Sundarbans,0.856
5,6,Which currency is used in Bangladesh?,Bangladeshi Taka,0.856
6,7,What is the national animal of Bangladesh?,Royal Bengal Tiger,0.825


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

,No,Question,Answer,Confidence
0,1,What is the capital city of Bangladesh?,Dhaka,0.957
1,2,Which river is considered the lifeline of Bangladesh?,Padma River,0.828
2,3,In which year did Bangladesh gain independence?,1971,0.993
3,4,What is the official language of Bangladesh?,Bengali,0.977
4,5,What is the name of the world’s largest mangrove forest located in Bangladesh?,Sundarbans,0.856
5,6,Which currency is used in Bangladesh?,Bangladeshi Taka,0.828
6,7,What is the national animal of Bangladesh?,Royal Bengal Tiger,0.825


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== Both PDFs combined =====


,No,Question,Answer,Confidence
0,1,What is the capital city of Bangladesh?,Dhaka,0.957
1,2,Which river is considered the lifeline of Bangladesh?,Padma River,0.828
2,3,In which year did Bangladesh gain independence?,1971,0.993
3,4,What is the official language of Bangladesh?,Bengali,0.977
4,5,What is the name of the world’s largest mangrove forest located in Bangladesh?,Sundarbans,0.795
5,6,Which currency is used in Bangladesh?,Bangladeshi Taka,0.856
6,7,What is the national animal of Bangladesh?,Royal Bengal Tiger,0.825


,Question,Easy PDF,Hard PDF,Combined
0,What is the capital city of Bangladesh?,Dhaka,Dhaka,Dhaka
1,Which river is considered the lifeline of Bangladesh?,Padma River,Padma River,Padma River
2,In which year did Bangladesh gain independence?,1971,1971,1971
3,What is the official language of Bangladesh?,Bengali,Bengali,Bengali
4,What is the name of the world’s largest mangrove forest located in Bangladesh?,Sundarbans,Sundarbans,Sundarbans
5,Which currency is used in Bangladesh?,Bangladeshi Taka,Bangladeshi Taka,Bangladeshi Taka
6,What is the national animal of Bangladesh?,Royal Bengal Tiger,Royal Bengal Tiger,Royal Bengal Tiger


## 8. Interactive mode
Ask your own question about the PDFs. Type `exit` to stop.

In [61]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Initialize the QA tool
tool = PDFQuestionAnswerer([EASY_PDF, HARD_PDF])

# Header Card
header = widgets.HTML("""
<div style='background: linear-gradient(135deg, #1a73e8 0%, #0d47a1 100%); padding: 20px; border-radius: 12px; color: white; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;'>
    <h2 style='margin: 0; font-weight: 600; font-size: 1.6em;'>📝 Smart PDF QA Assistant</h2>
    <p style='margin: 8px 0 0 0; opacity: 0.85; font-size: 0.95em; line-height: 1.4;'>
        Ask any questions about the loaded documents. The system will search across all documents using semantic retrieval and find the exact answer span.
    </p>
</div>
""")

# Text Input Box
query_input = widgets.Text(
    placeholder='Ask something (e.g., What is the world’s largest mangrove forest?)...',
    layout=widgets.Layout(width='75%', height='45px', margin='0 12px 0 0')
)

# Button with Custom Styling
submit_btn = widgets.Button(
    description='Get Answer',
    button_style='',
    icon='paper-plane',
    layout=widgets.Layout(width='22%', height='45px')
)
submit_btn.style.button_color = '#1a73e8'
submit_btn.style.text_color = 'white'

output_area = widgets.Output()

def get_score_badge(score):
    percent = int(score * 100)
    if score > 0.8:
        return f"<span style='background-color: #e6f4ea; color: #137333; padding: 4px 10px; border-radius: 20px; font-weight: bold; font-size: 0.9em; display: inline-flex; align-items: center;'>High Confidence ({percent}%)</span>"
    if score > 0.5:
        return f"<span style='background-color: #fef7e0; color: #b06000; padding: 4px 10px; border-radius: 20px; font-weight: bold; font-size: 0.9em; display: inline-flex; align-items: center;'>Medium Confidence ({percent}%)</span>"
    return f"<span style='background-color: #fce8e6; color: #c5221f; padding: 4px 10px; border-radius: 20px; font-weight: bold; font-size: 0.9em; display: inline-flex; align-items: center;'>Low Confidence ({percent}%)</span>"

def handle_submit(b):
    question = query_input.value.strip()
    if not question:
        return

    with output_area:
        clear_output()
        # Display a clean loading animation spinner or text
        display(HTML("""
        <div style='display: flex; align-items: center; gap: 10px; margin: 20px 0;'>
            <div style='border: 3px solid #f3f3f3; border-top: 3px solid #1a73e8; border-radius: 50%; width: 20px; height: 20px; animation: spin 1s linear infinite;'></div>
            <style>@keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }</style>
            <span style='color: #5f6368; font-family: sans-serif; font-size: 0.95em;'>Analyzing passages & extracting answers...</span>
        </div>
        """))

        try:
            res = tool.answer(question)
            ans = res.get("answer", "No clear answer could be extracted.")
            score = res.get("score", 0.0)
            context = res.get("context", "No context passage matched.")

            badge = get_score_badge(score)

            # Clean snippet logic processed out-of-line to avoid indentation errors
            context_html = ""
            if context:
                context_html = f"""
                <div style='margin-top: 16px; background-color: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 4px solid #1a73e8;'>
                    <strong style='color: #3c4043; font-size: 0.9em; display: block; margin-bottom: 6px;'>Retrieved Context Block:</strong>
                    <p style='margin: 0; font-style: italic; color: #4a3c3c; line-height: 1.5; font-size: 0.95em;'>"{context}"</p>
                </div>
                """

            # Structured card presentation
            result_html = f"""
            <div style='background: white; border: 1px solid #dadce0; border-radius: 12px; padding: 24px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;'>
                <div style='margin-bottom: 16px;'>
                    <span style='font-size: 0.85em; text-transform: uppercase; letter-spacing: 0.8px; color: #70757a; font-weight: bold;'>Matched Answer</span>
                </div>

                <h1 style='margin: 0 0 16px 0; color: #1a73e8; font-weight: bold; font-size: 1.85em;'>{ans}</h1>

                <div style='margin-bottom: 20px;'>
                    {badge}
                </div>

                <hr style='border: none; border-top: 1px solid #e8eaed; margin: 20px 0;' />

                <div style='margin-bottom: 12px;'>
                    <strong style='color: #3c4043; font-size: 0.95em;'>Question Asked:</strong>
                    <p style='margin: 4px 0 0 0; color: #5f6368; font-size: 0.95em;'>{question}</p>
                </div>

                {context_html}
            </div>
            """
            clear_output()
            display(HTML(result_html))
        except Exception as e:
            clear_output()
            display(HTML(f"<div style='background-color: #fce8e6; color: #c5221f; padding: 15px; border-radius: 8px; font-weight: bold; margin-top: 15px;'>Error: {e}</div>"))

submit_btn.on_click(handle_submit)
query_input.on_submit(handle_submit)

# Compose layouts with clean padding
input_bar = widgets.HBox([query_input, submit_btn], layout=widgets.Layout(align_items='center', margin='10px 0 20px 0'))
ui = widgets.VBox([
    header,
    input_bar,
    output_area
], layout=widgets.Layout(padding='10px'))

display(ui)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]